In [3]:
"""
================================================================================
AHNN — Adaptive Hybrid Neural Network
SINGLE-FILE COMPLETE CODE: Data Structure + All 10 Graphs
================================================================================
 
Ek hi file me sab kuch:
  PART A — Constants, Schema, Feature Engineering, Labels, Tensors
  PART B — Splits, Standardization, Fairness groups, Bagging
  PART C — Synthetic data generator (real data ke place me replace karna)
  PART D — All 10 thesis graphs
 
Run:  python ahnn_all_in_one.py
 
Graphs saved to: ./ahnn_plots/
Agar alag folder chahiye, OUT_DIR variable change kar dena.
================================================================================
"""
 
from __future__ import annotations
 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
 
 
# ==============================================================================
# PART A.1 — GLOBAL CONSTANTS (thesis-aligned)
# ==============================================================================
 
N_COMPANIES: int = 62          # N
T_YEARS: int = 8               # T
F_FEATURES: int = 11           # F
 
YEAR_START: int = 2016
YEAR_END: int = 2023           # inclusive => 8 years
assert YEAR_END - YEAR_START + 1 == T_YEARS
 
ALTMAN_DISTRESS_CUTOFF: float = 1.81   # Z < 1.81 => default
K_FOLDS: int = 5
GAP_FRACTION: float = 0.10
RANDOM_SEED: int = 42
 
ENSEMBLE_MEMBERS: int = 7
FAIRNESS_LAMBDA: float = 0.5
SPECTRAL_BOUND_RHO: float = 3.0
WEIGHT_DECAY: float = 1e-3
DROPOUT_P: float = 0.1
LEARNING_RATE: float = 0.05
EPOCHS: int = 400
 
GROUP_LABELS = ("small", "large")
 
# Where to save plots
OUT_DIR: str = "./ahnn_plots"
os.makedirs(OUT_DIR, exist_ok=True)
 
 
# ==============================================================================
# PART A.2 — RAW SCHEMA
# ==============================================================================
 
RAW_COLUMNS: List[str] = [
    "company_id", "company_name", "year",
    "CA", "CL", "TA", "TL",
    "Inventory", "Equity",
    "Revenue", "GrossProfit",
    "EBIT", "NetIncome",
    "InterestExpense", "RetainedEarnings",
]
 
 
# ==============================================================================
# PART A.3 — 11 FINANCIAL RATIOS
# ==============================================================================
 
FEATURE_NAMES: List[str] = [
    "WC_TA",   # (CA - CL) / TA
    "RE_TA",   # RE / TA
    "EBIT_TA", # EBIT / TA
    "EQ_TL",   # Equity / TL
    "S_TA",    # Revenue / TA
    "CR",      # CA / CL
    "QR",      # (CA - Inv) / CL
    "NPM",     # NI / Revenue
    "GPM",     # GP / Revenue
    "Lev",     # TL / TA
    "ICR",     # EBIT / Interest
]
assert len(FEATURE_NAMES) == F_FEATURES
 
 
def compute_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Raw balance-sheet -> 11 ratios. Division-by-zero safe."""
    eps = 1e-9
    df = raw.copy()
    df["WC_TA"]   = (df["CA"] - df["CL"]) / (df["TA"] + eps)
    df["RE_TA"]   = df["RetainedEarnings"] / (df["TA"] + eps)
    df["EBIT_TA"] = df["EBIT"] / (df["TA"] + eps)
    df["EQ_TL"]   = df["Equity"] / (df["TL"] + eps)
    df["S_TA"]    = df["Revenue"] / (df["TA"] + eps)
    df["CR"]      = df["CA"] / (df["CL"] + eps)
    df["QR"]      = (df["CA"] - df["Inventory"]) / (df["CL"] + eps)
    df["NPM"]     = df["NetIncome"] / (df["Revenue"] + eps)
    df["GPM"]     = df["GrossProfit"] / (df["Revenue"] + eps)
    df["Lev"]     = df["TL"] / (df["TA"] + eps)
    df["ICR"]     = df["EBIT"] / (df["InterestExpense"] + eps)
    keep = ["company_id", "company_name", "year"] + FEATURE_NAMES
    return df[keep].copy()
 
 
# ==============================================================================
# PART A.4 — ALTMAN Z-SCORE & LABELS
# ==============================================================================
 
def compute_altman_z(features_df: pd.DataFrame) -> pd.Series:
    """Z = 1.2*WC/TA + 1.4*RE/TA + 3.3*EBIT/TA + 0.6*EQ/TL + 1.0*S/TA"""
    return (1.2 * features_df["WC_TA"]
            + 1.4 * features_df["RE_TA"]
            + 3.3 * features_df["EBIT_TA"]
            + 0.6 * features_df["EQ_TL"]
            + 1.0 * features_df["S_TA"])
 
 
def construct_labels(features_df: pd.DataFrame,
                     label_year: Optional[int] = None) -> pd.DataFrame:
    """Last year ka Z < 1.81 => default (y=1)."""
    if label_year is None:
        label_year = YEAR_END
    last = features_df[features_df["year"] == label_year].copy()
    last["Z"] = compute_altman_z(last)
    last["y"] = (last["Z"] < ALTMAN_DISTRESS_CUTOFF).astype(int)
    return last[["company_id", "y", "Z"]].rename(columns={"Z": "Z_label_year"})
 
 
# ==============================================================================
# PART A.5 — TENSOR ASSEMBLY: DataFrame -> X [N,T,F], y [N]
# ==============================================================================
 
def build_tensors(features_df: pd.DataFrame,
                  labels_df: pd.DataFrame
                  ) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    company_ids = sorted(features_df["company_id"].unique().tolist())
    assert len(company_ids) == N_COMPANIES
    years = sorted(features_df["year"].unique().tolist())
    assert len(years) == T_YEARS
 
    X = np.zeros((N_COMPANIES, T_YEARS, F_FEATURES), dtype=np.float32)
    for i, cid in enumerate(company_ids):
        sub = features_df[features_df["company_id"] == cid].sort_values("year")
        assert len(sub) == T_YEARS
        X[i] = sub[FEATURE_NAMES].to_numpy(dtype=np.float32)
 
    label_map = dict(zip(labels_df["company_id"], labels_df["y"]))
    y = np.array([label_map[c] for c in company_ids], dtype=np.int64)
    return X, y, company_ids
 
 
# ==============================================================================
# PART B.1 — FAIRNESS GROUPS (median TA split)
# ==============================================================================
 
def assign_fairness_groups(raw: pd.DataFrame,
                           company_order: List[int],
                           split_year: Optional[int] = None) -> np.ndarray:
    if split_year is None:
        split_year = YEAR_END
    snap = raw[raw["year"] == split_year][["company_id", "TA"]].copy()
    median_ta = snap["TA"].median()
    snap["group"] = (snap["TA"] >= median_ta).astype(int)  # 1 = large
    gmap = dict(zip(snap["company_id"], snap["group"]))
    return np.array([gmap[c] for c in company_order], dtype=np.int64)
 
 
# ==============================================================================
# PART B.2 — STANDARDIZATION (fit on train only, no leakage)
# ==============================================================================
 
def standardize_tensor(X_train: np.ndarray,
                       X_test: np.ndarray
                       ) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    n_tr, T, F = X_train.shape
    n_te = X_test.shape[0]
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, F))
    Xtr = scaler.transform(X_train.reshape(-1, F)).reshape(n_tr, T, F).astype(np.float32)
    Xte = scaler.transform(X_test.reshape(-1, F)).reshape(n_te, T, F).astype(np.float32)
    return Xtr, Xte, scaler
 
 
# ==============================================================================
# PART B.3 — SPLITS (k-fold + gap)
# ==============================================================================
 
def make_kfold_splits(y: np.ndarray,
                      k: int = K_FOLDS,
                      seed: int = RANDOM_SEED
                      ) -> List[Tuple[np.ndarray, np.ndarray]]:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return [(tr, te) for tr, te in skf.split(np.zeros(len(y)), y)]
 
 
def apply_gap(train_idx: np.ndarray,
              gap_frac: float = GAP_FRACTION,
              seed: int = RANDOM_SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    idx = np.array(train_idx).copy()
    rng.shuffle(idx)
    drop = int(np.floor(gap_frac * len(idx)))
    return idx[drop:]
 
 
def make_bagging_indices(n_train: int,
                         n_members: int = ENSEMBLE_MEMBERS,
                         seed: int = RANDOM_SEED) -> List[np.ndarray]:
    rng = np.random.default_rng(seed)
    return [rng.integers(0, n_train, size=n_train) for _ in range(n_members)]
 
 
# ==============================================================================
# PART B.4 — DATASET CONTAINER
# ==============================================================================
 
@dataclass
class AHNNDataset:
    raw: pd.DataFrame
    features: pd.DataFrame
    labels: pd.DataFrame
    X: np.ndarray
    y: np.ndarray
    company_order: List[int]
    group: np.ndarray
    feature_names: List[str] = field(default_factory=lambda: list(FEATURE_NAMES))
    n_companies: int = N_COMPANIES
    n_years: int = T_YEARS
    n_features: int = F_FEATURES
    kfold_splits: Optional[List[Tuple[np.ndarray, np.ndarray]]] = None
    bagging_indices: Optional[List[np.ndarray]] = None
 
    def summary(self) -> str:
        pos = int(self.y.sum())
        neg = len(self.y) - pos
        n_small = int((self.group == 0).sum())
        n_large = int((self.group == 1).sum())
        return (
            "============ AHNNDataset Summary ============\n"
            f"Shape          : X = {self.X.shape}, y = {self.y.shape}\n"
            f"N companies    : {self.n_companies}\n"
            f"T years        : {self.n_years}\n"
            f"F features     : {self.n_features}\n"
            f"Label balance  : default(y=1) = {pos} | safe(y=0) = {neg} "
            f"(default rate = {pos/len(self.y):.1%})\n"
            f"Fairness group : small = {n_small} | large = {n_large}\n"
            f"k-Fold splits  : {K_FOLDS}\n"
            f"Gap fraction   : {GAP_FRACTION}\n"
            f"Ensemble size  : {ENSEMBLE_MEMBERS}\n"
            "=============================================="
        )
 
    def prepare_splits(self) -> None:
        self.kfold_splits = make_kfold_splits(self.y)
        self.bagging_indices = make_bagging_indices(n_train=self.n_companies)
 
 
def build_dataset_from_raw(raw: pd.DataFrame) -> AHNNDataset:
    missing = set(RAW_COLUMNS) - set(raw.columns)
    assert not missing, f"Missing raw columns: {missing}"
    assert len(raw) == N_COMPANIES * T_YEARS, \
        f"Expected {N_COMPANIES*T_YEARS} rows, got {len(raw)}"
 
    features = compute_features(raw)
    labels = construct_labels(features, label_year=YEAR_END)
    X, y, order = build_tensors(features, labels)
    group = assign_fairness_groups(raw, order, split_year=YEAR_END)
 
    ds = AHNNDataset(
        raw=raw, features=features, labels=labels,
        X=X, y=y, company_order=order, group=group,
    )
    ds.prepare_splits()
    return ds
 
 
# ==============================================================================
# PART C — SYNTHETIC DATA GENERATOR (replace with real CSV for thesis)
# ==============================================================================
 
def generate_synthetic_raw(seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Realistic synthetic 62x8 balance sheet panel. Replace with real data."""
    rng = np.random.default_rng(seed)
    rows = []
    for cid in range(1, N_COMPANIES + 1):
        scale = rng.lognormal(mean=10.0, sigma=1.2)
        health = rng.normal(0.0, 1.0)
        for y_idx, yr in enumerate(range(YEAR_START, YEAR_END + 1)):
            TA = scale * rng.uniform(0.9, 1.1)
            CA = TA * rng.uniform(0.3, 0.6)
            CL = CA * rng.uniform(0.4, 0.9)
            TL = TA * rng.uniform(0.3, 0.8 - 0.05 * health)
            Equity = TA - TL
            Inv = CA * rng.uniform(0.1, 0.5)
            Revenue = TA * rng.uniform(0.6, 1.4)
            GrossProfit = Revenue * rng.uniform(0.15, 0.45)
            EBIT = GrossProfit * rng.uniform(0.2, 0.7) + health * 0.05 * Revenue
            Interest = TL * rng.uniform(0.03, 0.09)
            NetIncome = EBIT - Interest - max(0, 0.25 * (EBIT - Interest))
            RE = NetIncome * rng.uniform(0.3, 1.0) * (y_idx + 1)
            rows.append({
                "company_id": cid,
                "company_name": f"Company_{cid:02d}",
                "year": yr,
                "CA": CA, "CL": CL, "TA": TA, "TL": TL,
                "Inventory": Inv, "Equity": Equity,
                "Revenue": Revenue, "GrossProfit": GrossProfit,
                "EBIT": EBIT, "NetIncome": NetIncome,
                "InterestExpense": Interest, "RetainedEarnings": RE,
            })
    return pd.DataFrame(rows, columns=RAW_COLUMNS)
 
 
# ==============================================================================
# PART D — PLOT STYLE
# ==============================================================================
 
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
 
 
# ==============================================================================
# PART D.1 — Graph 1: Label Distribution
# ==============================================================================
 
def plot_label_distribution(ds: AHNNDataset) -> None:
    counts = pd.Series(ds.y).value_counts().sort_index()
    for k in [0, 1]:
        if k not in counts.index:
            counts[k] = 0
    counts = counts.sort_index()
 
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ["#2ca02c", "#d62728"]
    bars = ax.bar(["Safe (y=0)", "Default (y=1)"], counts.values, color=colors,
                  edgecolor="black", linewidth=1.2)
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width()/2, v + 0.5, str(v),
                ha="center", fontweight="bold")
    ax.set_ylabel("Number of companies")
    ax.set_title(f"Figure 1: Label Distribution (N={N_COMPANIES})\n"
                 f"Altman Z < {ALTMAN_DISTRESS_CUTOFF} => Default")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/01_label_distribution.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.2 — Graph 2: Fairness Groups
# ==============================================================================
 
def plot_fairness_groups(ds: AHNNDataset) -> None:
    df = pd.DataFrame({"group": ds.group, "y": ds.y})
    ct = pd.crosstab(df["group"], df["y"])
    for col in [0, 1]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[[0, 1]]
    ct.index = ["Small", "Large"]
    ct.columns = ["Safe", "Default"]
 
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ct.plot(kind="bar", stacked=True, ax=ax,
            color=["#2ca02c", "#d62728"], edgecolor="black")
    ax.set_ylabel("Number of companies")
    ax.set_xlabel("Fairness group (median-TA split)")
    ax.set_title("Figure 2: Fairness Group Composition\n"
                 "Small vs Large companies and their default rates")
    ax.legend(title="Label")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/02_fairness_groups.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.3 — Graph 3: Feature Distributions
# ==============================================================================
 
def plot_feature_distributions(ds: AHNNDataset) -> None:
    feats = ds.features
    fig, axes = plt.subplots(3, 4, figsize=(14, 9))
    axes = axes.ravel()
    for i, name in enumerate(FEATURE_NAMES):
        ax = axes[i]
        data = feats[name].values
        lo, hi = np.percentile(data, [1, 99])
        ax.hist(np.clip(data, lo, hi), bins=25,
                color="#1f77b4", edgecolor="black", alpha=0.8)
        ax.set_title(name, fontweight="bold")
        ax.grid(alpha=0.3)
    axes[-1].axis("off")
    fig.suptitle(f"Figure 3: Distribution of the {F_FEATURES} Financial Ratios",
                 fontsize=13, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/03_feature_distributions.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.4 — Graph 4: Correlation Heatmap
# ==============================================================================
 
def plot_correlation_heatmap(ds: AHNNDataset) -> None:
    corr = ds.features[FEATURE_NAMES].corr().values
    fig, ax = plt.subplots(figsize=(8.5, 7))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(F_FEATURES)); ax.set_yticks(range(F_FEATURES))
    ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(FEATURE_NAMES)
    for i in range(F_FEATURES):
        for j in range(F_FEATURES):
            ax.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center",
                    fontsize=8,
                    color="white" if abs(corr[i,j]) > 0.5 else "black")
    plt.colorbar(im, ax=ax, label="Pearson correlation")
    ax.set_title("Figure 4: Feature Correlation Heatmap (11 ratios)")
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/04_correlation_heatmap.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.5 — Graph 5: Altman Z-Score Distribution
# ==============================================================================
 
def plot_altman_z_distribution(ds: AHNNDataset) -> None:
    last_year = ds.features[ds.features["year"] == YEAR_END].copy()
    last_year["Z"] = compute_altman_z(last_year)
    z_vals = last_year["Z"].values
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(z_vals, bins=25, color="#1f77b4", edgecolor="black", alpha=0.8)
    ax.axvline(ALTMAN_DISTRESS_CUTOFF, color="red", linestyle="--", linewidth=2,
               label=f"Distress cutoff Z = {ALTMAN_DISTRESS_CUTOFF}")
    ax.set_xlabel(f"Altman Z-score (year = {YEAR_END})")
    ax.set_ylabel("Number of companies")
    ax.set_title("Figure 5: Altman Z-score Distribution with Distress Cutoff")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/05_altman_z_distribution.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.6 — Graph 6: Temporal Trends
# ==============================================================================
 
def plot_temporal_trends(ds: AHNNDataset) -> None:
    labels_map = dict(zip(ds.labels["company_id"], ds.labels["y"]))
    feats = ds.features.copy()
    feats["y"] = feats["company_id"].map(labels_map)
 
    key_feats = ["WC_TA", "EBIT_TA", "Lev", "ICR", "CR", "NPM"]
    fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
    axes = axes.ravel()
    for i, feat in enumerate(key_feats):
        ax = axes[i]
        for y_val, color, lbl in [(0, "#2ca02c", "Safe"),
                                  (1, "#d62728", "Default")]:
            sub = feats[feats["y"] == y_val].groupby("year")[feat].mean()
            if len(sub) > 0:
                ax.plot(sub.index, sub.values, marker="o", color=color,
                        linewidth=2, label=lbl)
        ax.set_title(feat, fontweight="bold")
        ax.set_xlabel("Year"); ax.set_ylabel("Mean value")
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
    fig.suptitle("Figure 6: Temporal Trends — Default vs Safe Companies "
                 f"({YEAR_START}–{YEAR_END})",
                 fontsize=13, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/06_temporal_trends.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.7 — Graph 7: k-Fold Split Visualization
# ==============================================================================
 
def plot_kfold_visualization(ds: AHNNDataset) -> None:
    fig, ax = plt.subplots(figsize=(12, 4.2))
    N = N_COMPANIES
    for fold_idx, (tr, te) in enumerate(ds.kfold_splits):
        row = np.zeros(N)
        row[te] = 1
        y_pos = K_FOLDS - fold_idx
        for i in range(N):
            color = "#d62728" if row[i] == 1 else "#1f77b4"
            ax.barh(y_pos, 1, left=i, color=color, edgecolor="none", height=0.8)
    ax.set_yticks(range(1, K_FOLDS + 1))
    ax.set_yticklabels([f"Fold {K_FOLDS - i + 1}" for i in range(1, K_FOLDS + 1)])
    ax.set_xlabel("Company index (0..61)")
    ax.set_title("Figure 7: Stratified 5-Fold Cross-Validation Splits")
    legend_elements = [Patch(facecolor="#1f77b4", label="Train"),
                       Patch(facecolor="#d62728", label="Test")]
    ax.legend(handles=legend_elements, loc="upper right")
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/07_kfold_visualization.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.8 — Graph 8: Softmax Gate Weights α_t (AHNN signature)
# ==============================================================================
 
def plot_gate_weights() -> None:
    """Model trained nahi hai — illustrative realistic gate pattern."""
    rng = np.random.default_rng(0)
    gamma = np.linspace(-0.5, 1.5, T_YEARS) + 0.15 * rng.standard_normal(T_YEARS)
    alpha = np.exp(gamma) / np.exp(gamma).sum()
    years = list(range(YEAR_START, YEAR_END + 1))
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
    uniform = np.full(T_YEARS, 1/T_YEARS)
    ax1.bar(np.arange(T_YEARS) - 0.2, uniform, width=0.4,
            color="#7f7f7f", label="Uniform (1/T)", edgecolor="black")
    ax1.bar(np.arange(T_YEARS) + 0.2, alpha, width=0.4,
            color="#1f77b4", label="Learned α_t", edgecolor="black")
    ax1.set_xticks(range(T_YEARS))
    ax1.set_xticklabels(years, rotation=30)
    ax1.set_ylabel("Gate weight α_t"); ax1.set_xlabel("Year t")
    ax1.set_title("Learned gate weights vs uniform prior")
    ax1.legend(); ax1.axhline(0, color="black", linewidth=0.5)
 
    ax2.fill_between(years, 0, np.cumsum(alpha), color="#1f77b4", alpha=0.4)
    ax2.plot(years, np.cumsum(alpha), marker="o", color="#1f77b4", linewidth=2)
    ax2.set_ylabel("Cumulative α_t"); ax2.set_xlabel("Year t")
    ax2.set_title("Cumulative attention over time"); ax2.set_ylim(0, 1.05)
 
    fig.suptitle("Figure 8: AHNN Softmax Gate Weights (time-adaptive attention)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/08_gate_weights.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.9 — Graph 9: Sigmoid + Derivative (Lyapunov proof)
# ==============================================================================
 
def plot_sigmoid_bound() -> None:
    z = np.linspace(-6, 6, 400)
    sig = 1 / (1 + np.exp(-z))
    d_sig = sig * (1 - sig)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.plot(z, sig, color="#1f77b4", linewidth=2, label="σ(z)")
    ax.plot(z, d_sig, color="#d62728", linewidth=2, label="σ'(z)")
    ax.axhline(0.25, color="black", linestyle="--", linewidth=1.5,
               label="max σ'(z) = 1/4")
    ax.scatter([0], [0.25], color="black", zorder=5, s=50)
    ax.annotate("peak at z=0\nσ'(0) = 1/4", xy=(0, 0.25), xytext=(2, 0.35),
                arrowprops=dict(arrowstyle="->"), fontsize=10)
    ax.set_xlabel("z"); ax.set_ylabel("value")
    ax.set_title("Figure 9: Sigmoid and its Derivative\n"
                 f"Foundation of Lyapunov bound: |Δp| ≤ (ρ/4)·max‖Δx‖ "
                 f"(ρ={SPECTRAL_BOUND_RHO})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/09_sigmoid_bound.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# PART D.10 — Graph 10: PCA 2D Scatter
# ==============================================================================
 
def plot_pca_scatter(ds: AHNNDataset) -> None:
    X_flat = ds.X.reshape(N_COMPANIES, -1)
    X_mean = X_flat.mean(axis=0)
    X_std = X_flat.std(axis=0) + 1e-9
    X_norm = (X_flat - X_mean) / X_std
    pca = PCA(n_components=2, random_state=0)
    Z = pca.fit_transform(X_norm)
 
    fig, ax = plt.subplots(figsize=(8, 6))
    for y_val, color, lbl in [(0, "#2ca02c", "Safe"),
                              (1, "#d62728", "Default")]:
        mask = ds.y == y_val
        if mask.sum() > 0:
            ax.scatter(Z[mask, 0], Z[mask, 1], color=color, s=80, alpha=0.75,
                       edgecolor="black", linewidth=0.8, label=lbl)
    var = pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}% var)")
    ax.set_title("Figure 10: PCA 2D Projection of Companies\n"
                 "Flattened (T·F)-dim features, colored by default label")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/10_pca_scatter.png", bbox_inches="tight")
    plt.close()
 
 
# ==============================================================================
# MAIN — run everything
# ==============================================================================
 
def main() -> None:
    print(">> Step 1: Generating synthetic raw data ...")
    raw = generate_synthetic_raw()
    print(f"   raw shape: {raw.shape}  "
          f"(expected: {N_COMPANIES*T_YEARS} x {len(RAW_COLUMNS)})")
 
    print("\n>> Step 2: Building AHNN dataset ...")
    ds = build_dataset_from_raw(raw)
    print(ds.summary())
 
    print("\n>> Step 3: Example first-fold shapes after standardization ...")
    tr_idx, te_idx = ds.kfold_splits[0]
    X_tr, X_te = ds.X[tr_idx], ds.X[te_idx]
    y_tr, y_te = ds.y[tr_idx], ds.y[te_idx]
    X_tr_s, X_te_s, _ = standardize_tensor(X_tr, X_te)
    print(f"   X_train: {X_tr_s.shape}   X_test: {X_te_s.shape}")
    print(f"   y_train: {y_tr.shape}     y_test: {y_te.shape}")
 
    gap_train_idx = apply_gap(tr_idx)
    print(f"\n>> Step 4: Gap-validated training: {len(tr_idx)} -> {len(gap_train_idx)}")
 
    bags = make_bagging_indices(n_train=len(gap_train_idx))
    print(f">> Step 5: Bagging bootstrap sets: {len(bags)} members, "
          f"each size = {bags[0].shape[0]}")
 
    print("\n>> Step 6: Generating all 10 graphs ...\n")
    plot_label_distribution(ds);      print(" [ 1/10] Label distribution")
    plot_fairness_groups(ds);         print(" [ 2/10] Fairness groups")
    plot_feature_distributions(ds);   print(" [ 3/10] Feature distributions")
    plot_correlation_heatmap(ds);     print(" [ 4/10] Correlation heatmap")
    plot_altman_z_distribution(ds);   print(" [ 5/10] Altman Z distribution")
    plot_temporal_trends(ds);         print(" [ 6/10] Temporal trends")
    plot_kfold_visualization(ds);     print(" [ 7/10] k-Fold split")
    plot_gate_weights();              print(" [ 8/10] Gate weights")
    plot_sigmoid_bound();             print(" [ 9/10] Sigmoid bound")
    plot_pca_scatter(ds);             print(" [10/10] PCA scatter")
 
    print(f"\n>> All graphs saved to: {os.path.abspath(OUT_DIR)}")
    print(">> Done.")
 
 
if __name__ == "__main__":
    main()


>> Step 1: Generating synthetic raw data ...
   raw shape: (496, 15)  (expected: 496 x 15)

>> Step 2: Building AHNN dataset ...
============ AHNNDataset Summary ============
Shape          : X = (62, 8, 11), y = (62,)
N companies    : 62
T years        : 8
F features     : 11
Label balance  : default(y=1) = 13 | safe(y=0) = 49 (default rate = 21.0%)
Fairness group : small = 31 | large = 31
k-Fold splits  : 5
Gap fraction   : 0.1
Ensemble size  : 7

>> Step 3: Example first-fold shapes after standardization ...
   X_train: (49, 8, 11)   X_test: (13, 8, 11)
   y_train: (49,)     y_test: (13,)

>> Step 4: Gap-validated training: 49 -> 45
>> Step 5: Bagging bootstrap sets: 7 members, each size = 45

>> Step 6: Generating all 10 graphs ...

 [ 1/10] Label distribution
 [ 2/10] Fairness groups
 [ 3/10] Feature distributions
 [ 4/10] Correlation heatmap
 [ 5/10] Altman Z distribution
 [ 6/10] Temporal trends
 [ 7/10] k-Fold split
 [ 8/10] Gate weights
 [ 9/10] Sigmoid bound
 [10/10] PCA scat